# 🎙️ Lecture Q&A with Audio Playback

**Pipeline:** Audio file or URL → Parakeet TDT 0.6B v3 ASR (with timestamps) → FAISS semantic search → LLaMA 3.1 8B answers → play back the relevant clip inline.

> ⚠️ **Runtime → Change runtime type → T4 GPU** before running.

## ⚙️ Configuration — Edit Here First

Set either a **local file path** or a **URL** in `AUDIO_SOURCE`.  
If a URL is provided the file is downloaded automatically before transcription.

In [ ]:
# ─── USER CONFIGURATION ───────────────────────────────────────────────────────

# Set to a local file path OR a direct URL to a .wav file
AUDIO_SOURCE = 'https://www.uclass.psychol.ucl.ac.uk/Release2/Conversation/AudioOnly/wav/M_0048_11y6m_2.wav'

# Local filename to save to (used when AUDIO_SOURCE is a URL)
AUDIO_FILE = 'input.wav'

NVIDIA_API_KEY = 'YOUR_NVIDIA_API_KEY_HERE'  # ← paste your NVIDIA API key here

# ──────────────────────────────────────────────────────────────────────────────

import os
os.environ['NVIDIA_API_KEY'] = NVIDIA_API_KEY
print(f'Audio source : {AUDIO_SOURCE}')
print(f'API key set  : {"yes" if NVIDIA_API_KEY != "YOUR_API_KEY_HERE" else "NO – please set NVIDIA_API_KEY above"}')

## Step 0 — Install Dependencies

Run once per session. The restart-check cell that follows reboots automatically if the numpy pin needs applying.

In [2]:
!pip install -q 'nemo_toolkit[asr]'
!pip install -q sentence-transformers faiss-cpu langchain langchain-nvidia-ai-endpoints pydub ipywidgets
!pip install -U lhotse -q                                                    # latest lhotse (NeMo requires is_valid_url)
!pip install -q 'numpy==1.26.4' 'scipy==1.13.1' --force-reinstall           # pin AFTER other installs so it wins
print('✓ All dependencies installed')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 29.6 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 13.1 MB/s eta 0:00:0000:0100:01m
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
blosc2 4.0.0 requires numexpr>=2.14.1; platform_machine != "wasm32", but you have numexpr 2.13.1 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.2.6 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.1 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.1 which is incom

In [3]:
# Auto-restart kernel if numpy version does not match the pin
import numpy as np
if np.__version__ != '1.26.4':
    print(f'numpy {np.__version__} detected → restarting kernel to apply pin …')
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)
else:
    print(f'✓ numpy {np.__version__} — ready, continue')

✓ numpy 1.26.4 — ready, continue


## Step 0b — Imports

In [4]:
import os, io, re, types, warnings, urllib.request
import numpy as np
import scipy.io.wavfile as wav_io   # always imported; used as pydub fallback
import faiss
from typing import List, Dict
from sentence_transformers import SentenceTransformer
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.messages import HumanMessage, SystemMessage
from IPython.display import display, Audio, HTML

# pydub — preferred for audio slicing; scipy is the fallback
try:
    from pydub import AudioSegment
    PYDUB_OK = True
except Exception:
    PYDUB_OK = False

warnings.filterwarnings('ignore')
print('✓ Imports complete')
print(f'  pydub available : {PYDUB_OK}')
print(f'  numpy version   : {np.__version__}')

✓ Imports complete
  pydub available : True
  numpy version   : 1.26.4


## Step 1 — Resolve Audio Input

* If `AUDIO_SOURCE` is a **URL** → download it to `AUDIO_FILE`.
* If `AUDIO_SOURCE` is a **local path** → use it directly.
* **Google Colab upload** option is also provided (commented out).

In [5]:
# ── Fallback: define config vars if the config cell was skipped/kernel restarted
import os, re, urllib.request
try:
    AUDIO_SOURCE
except NameError:
    AUDIO_SOURCE = 'https://www.uclass.psychol.ucl.ac.uk/Release2/Conversation/AudioOnly/wav/M_0048_11y6m_2.wav'
    AUDIO_FILE   = 'input.wav'
    print('⚠️  Config cell was not run — using built-in defaults.')
    print('   To use your own file/key, run the config cell first, then re-run this cell.')

try:
    NVIDIA_API_KEY
except NameError:
    NVIDIA_API_KEY = os.environ.get('NVIDIA_API_KEY', 'YOUR_API_KEY_HERE')

os.environ['NVIDIA_API_KEY'] = NVIDIA_API_KEY

# ── Option A: Google Colab manual upload (uncomment to use) ──────────────────
# try:
#     from google.colab import files
#     uploaded = files.upload()               # opens a file picker
#     AUDIO_FILE = next(iter(uploaded))       # first uploaded filename
#     print(f'✓ Uploaded: {AUDIO_FILE}')
# except ImportError:
#     print('Not in Colab — using AUDIO_SOURCE from config cell.')

# ── Option B: URL or local path (automatic) ──────────────────────────────────
def _is_url(s: str) -> bool:
    return re.match(r'^https?://', s.strip()) is not None

if _is_url(AUDIO_SOURCE):
    if os.path.exists(AUDIO_FILE):
        print(f'✓ Audio already downloaded: {AUDIO_FILE}  (skipping re-download)')
    else:
        print(f'Downloading {AUDIO_SOURCE} …')
        urllib.request.urlretrieve(AUDIO_SOURCE, AUDIO_FILE)
        print(f'✓ Saved to {AUDIO_FILE}')
else:
    AUDIO_FILE = AUDIO_SOURCE   # treat as local path
    print(f'Using local file: {AUDIO_FILE}')

# ── Sanity check ─────────────────────────────────────────────────────────────
if not os.path.exists(AUDIO_FILE):
    raise FileNotFoundError(
        f'Audio file not found: {AUDIO_FILE}\n'
        f'Check AUDIO_SOURCE in the config cell or use the Colab upload option.'
    )

size_mb = os.path.getsize(AUDIO_FILE) / (1024 ** 2)
print(f'✓ Audio file ready: {AUDIO_FILE}  ({size_mb:.1f} MB)')

✓ Audio already downloaded: input.wav  (skipping re-download)
✓ Audio file ready: input.wav  (27.3 MB)


## Step 2a — CUDA Compatibility Patch

CUDA 13 changed `cudaStreamGetCaptureInfo` to return 5 values; NeMo's TDT decoder expects 6.  
`torch.cuda.get_device_capability` is also wrapped to avoid driver-level exceptions.

In [6]:
import torch

# Patch get_device_capability so it never throws
_orig_cap = torch.cuda.get_device_capability
def _safe_get_device_capability(device=None):
    try:
        return _orig_cap(device)
    except Exception:
        return (8, 0)          # pretend Ampere if driver call fails
torch.cuda.get_device_capability = _safe_get_device_capability

print('✓ CUDA patch applied')
print(f'  CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  Device         : {torch.cuda.get_device_name(0)}')
    print(f'  Capability     : {torch.cuda.get_device_capability(0)}')

✓ CUDA patch applied
  CUDA available : False


## Step 2b — Load ASR Model

**NVIDIA Parakeet TDT 0.6B v3** (~2.5 GB download on first run).

In [7]:
import nemo.collections.asr as nemo_asr
from nemo.collections.asr.parts.submodules.transducer_decoding.tdt_label_looping import (
    GreedyBatchedTDTLabelLoopingComputer,
)

asr_model = nemo_asr.models.ASRModel.from_pretrained('nvidia/parakeet-tdt-0.6b-v3')

# Force NO_GRAPHS mode to work around CUDA 13 stream-capture ABI change
_NO_GRAPHS = GreedyBatchedTDTLabelLoopingComputer.CudaGraphsMode.NO_GRAPHS
_orig_decoding = asr_model.change_decoding_strategy.__func__

def _patched_decoding(self, decoding_cfg, verbose=True):
    _orig_decoding(self, decoding_cfg, verbose)
    try:
        dc = self.decoding.decoding.decoding_computer
        if hasattr(dc, 'cuda_graphs_mode'):
            dc.cuda_graphs_mode = _NO_GRAPHS
    except AttributeError:
        pass

asr_model.change_decoding_strategy = types.MethodType(_patched_decoding, asr_model)
asr_model.change_decoding_strategy(asr_model.cfg.decoding, verbose=False)

print('✓ ASR model ready')

[NeMo W 2026-03-09 22:08:07 megatron_init:62] Megatron num_microbatches_calculator not found, using Apex version.


parakeet-tdt-0.6b-v3.nemo:   0%|          | 0.00/2.51G [00:00<?, ?B/s]

[NeMo I 2026-03-09 22:09:18 mixins:184] Tokenizer SentencePieceTokenizer initialized with 8192 tokens


[NeMo W 2026-03-09 22:09:23 modelPT:188] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    use_lhotse: true
    skip_missing_manifest_entries: true
    input_cfg: null
    tarred_audio_filepaths: null
    manifest_filepath: null
    sample_rate: 16000
    shuffle: true
    num_workers: 2
    pin_memory: true
    max_duration: 10.0
    min_duration: 1.0
    text_field: answer
    batch_duration: null
    max_tps: null
    use_bucketing: true
    bucket_duration_bins: null
    bucket_batch_size: null
    num_buckets: 30
    bucket_buffer_size: 20000
    shuffle_buffer_size: 10000
    
[NeMo W 2026-03-09 22:09:23 modelPT:195] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation 

[NeMo I 2026-03-09 22:09:31 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}
[NeMo I 2026-03-09 22:09:31 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}


...

[NeMo I 2026-03-09 22:09:31 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}


[NeMo W 2026-03-09 22:09:31 label_looping_base:125] No conditional node support for Cuda.
    Cuda graphs with while loops are disabled, decoding speed will be slower
    Reason: CUDA is not available


[NeMo I 2026-03-09 22:09:35 save_restore_connector:285] Model EncDecRNNTBPEModel was successfully restored from /root/.cache/huggingface/hub/models--nvidia--parakeet-tdt-0.6b-v3/snapshots/6d590f77001d318fb17a0b5bf7ee329a91b52598/parakeet-tdt-0.6b-v3.nemo.
[NeMo I 2026-03-09 22:09:35 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}


[NeMo W 2026-03-09 22:09:35 label_looping_base:125] No conditional node support for Cuda.
    Cuda graphs with while loops are disabled, decoding speed will be slower
    Reason: CUDA is not available


✓ ASR model ready


## Step 2c — Transcribe Audio

Word-level and segment-level timestamps are enabled. The full transcript is printed when done.

In [8]:
print(f'Transcribing: {AUDIO_FILE} …')
asr_output = asr_model.transcribe([AUDIO_FILE], timestamps=True)

raw_segments    = asr_output[0].timestamp['segment']
word_timestamps = asr_output[0].timestamp['word']

# Build structured segment list
segments: List[Dict] = [
    {
        'segment_index': i,
        'text':          seg['segment'],
        'start_time':    float(seg['start']),
        'end_time':      float(seg['end']),
    }
    for i, seg in enumerate(raw_segments)
]

print(f'\n✓ Transcribed {len(segments)} segments\n')
print('=' * 70)
print('FULL TRANSCRIPT')
print('=' * 70)
for seg in segments:
    m_s, s_s = divmod(int(seg['start_time']), 60)
    m_e, s_e = divmod(int(seg['end_time']),   60)
    print(f"[{m_s:02d}:{s_s:02d} - {m_e:02d}:{s_e:02d}]  {seg['text']}")

Transcribing: input.wav …
[NeMo I 2026-03-09 22:09:35 rnnt_models:296] Timestamps requested, setting decoding timestamps to True. Capture them in Hypothesis object,                         with output[0][idx].timestep['word'/'segment'/'char']
[NeMo I 2026-03-09 22:09:35 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}


[NeMo W 2026-03-09 22:09:35 label_looping_base:125] No conditional node support for Cuda.
    Cuda graphs with while loops are disabled, decoding speed will be slower
    Reason: CUDA is not available
[NeMo W 2026-03-09 22:09:35 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-03-09 22:09:35 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
Transcribing: 1it [03:50, 230.61s/it]


✓ Transcribed 86 segments

FULL TRANSCRIPT
[00:00 - 00:02]  Tell me about Christmas and New Year.
[00:03 - 00:04]  What did you get up to?
[00:05 - 00:11]  I was working quite a lot during the Christmas and New Year, eating.
[00:12 - 00:17]  And I did quite a lot of cooking and helping in my parents' restaurant.
[00:19 - 00:25]  Okay, what uh what sort of stuff were you cooking described and how you were doing it?
[00:25 - 00:29]  I was cooking car was the Czechs, is that Czech tradition?
[00:29 - 00:29]  Car?
[00:30 - 00:31]  Yeah yes.
[00:31 - 00:32]  As in a fish.
[00:32 - 00:33]  Yes.
[00:33 - 00:35]  Bread bread crumbed.
[00:35 - 00:37]  And then pan fried.
[00:38 - 00:44]  It served with potato salad and Czech lager.
[00:45 - 00:46]  Czech lager.
[00:46 - 00:46]  Yes.
[00:49 - 00:49]  No.
[00:50 - 00:50]  Okay.
[00:50 - 00:53]  Um were you cooking anything else?
[00:56 - 00:57]  No.
[00:57 - 00:57]  No.
[00:58 - 00:58]  Okay.
[00:58 - 01:02]  Um tell me about what you did on Chr

## Step 3 — Build FAISS Vector Index

Each segment is embedded with `all-MiniLM-L6-v2` (384-dim) and stored in a flat L2 FAISS index.

In [9]:
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

texts      = [seg['text'] for seg in segments]
embeddings = embed_model.encode(texts, show_progress_bar=True).astype('float32')

faiss_index = faiss.IndexFlatL2(embeddings.shape[1])
faiss_index.add(embeddings)

print(f'\n✓ FAISS index built — {faiss_index.ntotal} segments indexed')


def search_segments(query: str, k: int = 3) -> List[Dict]:
    """Return the top-k most relevant segments for a natural-language query."""
    k = min(k, len(segments))
    q_emb = embed_model.encode([query]).astype('float32')
    _, idxs = faiss_index.search(q_emb, k)
    return [segments[i] for i in idxs[0]]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]


✓ FAISS index built — 86 segments indexed


## Step 4 — Q&A with Audio Playback

For each question:
1. Top **3** semantically relevant segments are retrieved from FAISS.
2. **LLaMA 3.1 8B** (via NVIDIA API) generates a grounded, timestamped answer.
3. The **single most relevant** segment is highlighted and its audio clip is played inline.

Type **`quit`** to exit the loop.

In [ ]:
import base64

# ─── Audio slicing ────────────────────────────────────────────────────────────

def slice_audio_pydub(filepath: str, start_s: float, end_s: float) -> bytes:
    """Slice with pydub and return raw WAV bytes."""
    audio = AudioSegment.from_file(filepath)
    clip  = audio[int(start_s * 1000): int(end_s * 1000)]
    buf   = io.BytesIO()
    clip.export(buf, format='wav')
    return buf.getvalue()


def slice_audio_scipy(filepath: str, start_s: float, end_s: float) -> bytes:
    """Slice with scipy and return WAV bytes."""
    import scipy.io.wavfile as wav_io
    import struct, wave
    sr, data = wav_io.read(filepath)
    clip = data[int(start_s * sr): int(end_s * sr)]
    buf = io.BytesIO()
    with wave.open(buf, 'wb') as wf:
        n_channels = 1 if clip.ndim == 1 else clip.shape[1]
        wf.setnchannels(n_channels)
        wf.setsampwidth(2)   # int16
        wf.setframerate(sr)
        wf.writeframes(clip.astype('int16').tobytes())
    return buf.getvalue()


def play_segment(seg: dict) -> None:
    """Slice the segment and display an inline HTML5 audio player (works in VS Code & Colab)."""
    start_buf = max(0.0, seg['start_time'] - 0.3)
    end_buf   = seg['end_time'] + 0.3
    try:
        if PYDUB_OK:
            wav_bytes = slice_audio_pydub(AUDIO_FILE, start_buf, end_buf)
        else:
            wav_bytes = slice_audio_scipy(AUDIO_FILE, start_buf, end_buf)

        b64 = base64.b64encode(wav_bytes).decode('utf-8')
        display(HTML(
            f'<audio controls style="width:100%;margin-top:6px">'
            f'<source src="data:audio/wav;base64,{b64}" type="audio/wav">'
            f'</audio>'
        ))
    except Exception as exc:
        print(f'  ⚠️  Audio playback failed: {exc}')


def fmt_ts(seg: dict) -> str:
    """Format start/end as MM:SS – MM:SS."""
    m_s, s_s = divmod(int(seg['start_time']), 60)
    m_e, s_e = divmod(int(seg['end_time']),   60)
    return f'{m_s:02d}:{s_s:02d} – {m_e:02d}:{s_e:02d}'


print('✓ Audio helpers ready')

# ─── LLM ─────────────────────────────────────────────────────────────────────

llm = ChatNVIDIA(
    model='meta/llama-3.1-8b-instruct',
    nvidia_api_key=os.environ.get('NVIDIA_API_KEY'),
    temperature=0.2,
    max_completion_tokens=512,
)
print('✓ LLM ready')

# ─── Q&A function ────────────────────────────────────────────────────────────

def answer_question(question: str, top_k: int = 3) -> None:
    hits = search_segments(question, k=top_k)

    context = '\n\n'.join(
        f"Segment {i+1} [{fmt_ts(s)}]:\n{s['text']}"
        for i, s in enumerate(hits)
    )

    msgs = [
        SystemMessage(content=(
            'You are a helpful assistant. Answer the user\'s question using '
            'only the provided transcript segments. Always cite the timestamp(s) '
            'of the segment(s) you draw from. Be concise and direct.'
        )),
        HumanMessage(content=f'Question: {question}\n\nTranscript segments:\n{context}'),
    ]

    response = llm.invoke(msgs).content

    print('\n' + '=' * 70)
    print('💡 ANSWER')
    print('-' * 70)
    print(response)

    best = hits[0]
    print('\n' + '-' * 70)
    print(f'📌 Most relevant segment  [{fmt_ts(best)}]')
    display(HTML(
        '<div style="background:#fffbcc;border-left:4px solid #f5c518;'
        'padding:8px 12px;border-radius:4px;font-family:monospace;">'
        + best['text'] +
        '</div>'
    ))

    print('🔊 Audio clip:')
    play_segment(best)

    print('\n📍 All retrieved segments:')
    for i, s in enumerate(hits):
        print(f"  {i+1}. [{fmt_ts(s)}] {s['text']}")
    print('=' * 70)


print('\n✓ Q&A system ready — run the next cell to start asking questions')

In [11]:
print('Ask a question about the lecture. Type "quit" to exit.\n')

while True:
    try:
        question = input('❓ Your question: ').strip()
    except EOFError:
        break

    if not question:
        continue

    if question.lower() in ('quit', 'exit', 'q'):
        print('👋 Goodbye!')
        break

    answer_question(question)

Ask a question about the lecture. Type "quit" to exit.


💡 ANSWER
----------------------------------------------------------------------
The topic of conversation is not explicitly clear, but it seems to be about school and personal items, possibly a school assignment or project.

----------------------------------------------------------------------
📌 Most relevant segment  [03:52 – 04:13]


🔊 Audio clip:



📍 All retrieved segments:
  1. [03:52 – 04:13] Um what about um about describe your bedroom?
  2. [02:01 – 02:03] Alright, tell me about uh what chapter at school.
  3. [02:11 – 02:12] Describe some of the stuff.
👋 Goodbye!
